In [1]:
import numpy as np
import pandas as pd
import os, warnings 
warnings.filterwarnings('ignore') 
from setup import *

In [2]:
thre_count = 15
def extract_specific_features(df_diff, df_counts, label, thre_diff=.3, thre_pval=3, ntop=5, nbottom=5):
    """
    Extract subtype-specific features based on significance, frequency, and differential expression.
    
    Parameters:
    df_diff: DataFrame with differential expression results (must contain 'subtype', label, 'neg_log_p_adj', 'diff')
    df_counts: DataFrame with count data (must contain 'subtype', label, 'count')
    label: str, the feature label column name (e.g., 'TF')
    thre_count: int, minimum count threshold for feature inclusion
    
    Returns:
    DataFrame with subtype-specific features meeting all criteria, including all samples for selected features
    """
    # Create a list to store results for each subtype
    subtype_specific_features = []
    
    # Get unique subtypes
    unique_subtypes = df_diff['subtype'].unique()
    
    for subtype in unique_subtypes:
        # First, get features that appear in at least thre_count samples for this subtype
        subtype_feature_counts = df_counts[
            (df_counts['subtype'] == subtype) & 
            (df_counts['count'] >= thre_count)
        ][label].tolist()
        
        if not subtype_feature_counts:
            continue
        
        # Now filter the differential expression dataframe for this subtype
        # Only filter by significance and count here, NOT by diff.abs()
        subtype_df = df_diff[
            (df_diff['subtype'] == subtype) & 
            (df_diff[label].isin(subtype_feature_counts)) &
            (df_diff['neg_log_p_adj'] > thre_pval)
        ].copy()
        
        if not subtype_df.empty:
            # Get top 3 features with largest diff (most upregulated)
            top_features = subtype_df.nlargest(ntop, 'diff')[label].unique().tolist()
            
            # Get bottom 2 features with smallest diff (most downregulated)
            bottom_features = subtype_df.nsmallest(nbottom, 'diff')[label].unique().tolist()
            
            # Combine and remove duplicates
            selected_features = list(set(top_features + bottom_features))
            
            if selected_features:
                # Get ALL samples for these selected features in this subtype
                # BUT now filter to only include samples where diff.abs() > 1
                selected_df = df_diff[
                    (df_diff['subtype'] == subtype) & 
                    (df_diff[label].isin(selected_features)) &
                    (df_diff['diff'].abs() > thre_diff)  # Apply the diff filter here
                ].copy()
                
                # Verify that each selected feature still appears in enough samples
                valid_features = []
                for feature in selected_features:
                    feature_count = len(selected_df[selected_df[label] == feature])
                    if feature_count >= thre_count:
                        valid_features.append(feature)
                
                if valid_features:
                    # Filter to only include features that still meet the count threshold
                    selected_df = selected_df[selected_df[label].isin(valid_features)]
                    subtype_specific_features.append(selected_df)
    
    # Combine all results into a single DataFrame
    if subtype_specific_features:
        final_result = pd.concat(subtype_specific_features, ignore_index=True)
        return final_result
    else:
        return pd.DataFrame()

In [3]:
def filter_rows(df_diff, final_result, key):
    existing_pairs = set(zip(final_result[key], final_result['subtype']))
    mask = df_diff.apply(lambda row: (row[key], row['subtype']) in existing_pairs, axis=1)
    filtered_df = df_diff[mask]
    return filtered_df

## MP-specific TFs

In [4]:
df_tf_diff = pd.read_csv('dataframes/across_metaprogram/tf_score.csv', index_col=0)
df_tf_counts = (
    df_tf_diff.groupby('subtype')['TF']
    .value_counts()
    .reset_index(name='count')
)

In [5]:
final_result_tfs = extract_specific_features(df_tf_diff, df_tf_counts, 'TF', thre_pval=3, thre_diff=0.3)
final_result_tfs = pd.concat(
    [final_result_tfs, 
     df_tf_diff[
         (df_tf_diff['subtype'] == 'Mac') & (df_tf_diff['TF'] == 'STAT3') ]],
    ignore_index=True)
final_result_tfs

,sample,TF,subtype,TFa,p_adj,neg_log_p_adj,rank,abs_TFa,diff
0,MGH258,SREBF2,AC,0.453035,1.453836e-04,3.837484,2,0.453035,0.476063
1,MGH258,SOX2,AC,0.298573,8.317063e-03,2.080030,26,0.298573,0.313750
2,MGH258,DBP,AC,-0.335331,1.908860e-03,2.719226,194,0.335331,-0.352376
3,MGH258,PAX6,AC,-0.290927,8.255773e-04,3.083242,196,0.290927,-0.305715
4,UKF243,ETV5,AC,-0.337541,5.561347e-12,9.976495,190,0.337541,-0.410720
...,...,...,...,...,...,...,...,...,...
1170,ZH881T1,STAT3,Mac,0.362491,6.011730e-07,6.220928,53,0.362491,0.412192
1171,ZH881inf,STAT3,Mac,-0.330262,2.963605e-03,2.528180,141,0.330262,-0.366061
1172,ZH916T1,STAT3,Mac,-0.091047,1.012770e-01,0.994489,140,0.091047,-0.101294
1173,ZH916bulk,STAT3,Mac,0.220474,1.645318e-01,0.783750,66,0.220474,0.227322


In [6]:
filtered_df = filter_rows(df_tf_diff, final_result_tfs, key = 'TF')
filtered_df

,sample,TF,subtype,TFa,p_adj,neg_log_p_adj,rank,abs_TFa,diff
6,MGH258,MEF2C,Mac,0.753543,5.640818e-46,10.000000,0,0.753543,0.872233
11,MGH258,NR3C1,Vasc,0.627694,1.130669e-44,10.000000,0,0.627694,0.741303
17,MGH258,HIF1A,MES.Hyp,0.886400,3.371764e-13,9.998538,1,0.886400,0.918798
18,MGH258,RFX3,Mac,0.763700,9.902436e-46,10.000000,1,0.763700,0.883990
24,MGH258,SREBF2,AC,0.453035,1.453836e-04,3.837484,2,0.453035,0.476063
...,...,...,...,...,...,...,...,...,...
75851,ZH916inf,MEIS1,Chromatin.Reg,-0.314950,1.458216e-02,1.836178,171,0.314950,-0.349459
75977,ZH916inf,HIF1A,Oligo,-0.357753,9.925902e-06,5.003226,180,0.357753,-0.432346
75994,ZH916inf,MAZ,Chromatin.Reg,-0.506564,4.498772e-05,4.346905,182,0.506564,-0.562069
76021,ZH916inf,ZNF84,Inflammatory.Mac,-1.215264,1.993485e-34,10.000000,184,1.215264,-1.394072


In [7]:
filtered_df.to_csv('dataframes/mp_specific_tfs.csv')

## MP-specific Pathways

In [ ]:
df_pw_diff = pd.read_csv('dataframes/across_metaprogram/pw_score.csv', index_col=0)
df_pw_diff['PW'] = [hallmarks[x] for x in df_pw_diff['PW']]
df_pw_counts = (
    df_pw_diff.groupby('subtype')['PW']
    .value_counts()
    .reset_index(name='count')
)

In [ ]:
final_result_pws = extract_specific_features(df_pw_diff, df_pw_counts, 'PW')
final_result_pws

In [ ]:
filtered_df = filter_rows(df_pw_diff, final_result_pws, key = 'PW')
filtered_df

In [ ]:
# final_result_pws.to_csv('dataframes/mp_specific_pws.csv')
filtered_df.to_csv('dataframes/mp_specific_pws.csv')

## MP-specific Drugs

In [ ]:
df_d2c_diff = pd.read_csv('dataframes/across_metaprogram/d2c_score.csv', index_col=0)
df_d2c_counts = (
    df_d2c_diff.groupby('subtype')['DC']
    .value_counts()
    .reset_index(name='count')
)

In [ ]:
final_result_dcs = extract_specific_features(df_d2c_diff, df_d2c_counts, 'DC', ntop=5, nbottom=0)
final_result_dcs

In [ ]:
filtered_df = filter_rows(df_d2c_diff, final_result_dcs, key = 'DC')
filtered_df

In [ ]:
# final_result_dcs.to_csv('dataframes/mp_specific_dcs.csv')
filtered_df.to_csv('dataframes/mp_specific_dcs.csv')